In [0]:
--CREATE SCHEMA IF NOT EXISTS main.bronze;

CREATE TABLE IF NOT EXISTS workspace.pyspark_training.orders_delta (
  order_id STRING,
  customer_id STRING,
  order_ts TIMESTAMP,
  item_id STRING,
  qty INT,
  price DOUBLE
)
USING DELTA;

In [0]:
select * from workspace.pyspark_training.orders_delta

In [0]:
CREATE VOLUME IF NOT EXISTS workspace.pyspark_training.raw_data

In [0]:
%python
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
)

catalog = "workspace"
schema  = "pyspark_training"
volume  = "raw_data"

incoming_path   = f"/Volumes/{catalog}/{schema}/{volume}/incoming"
checkpoint_dir  = f"/Volumes/{catalog}/{schema}/{volume}/_checkpoints/ingest_orders"
schema_loc      = f"/Volumes/{catalog}/{schema}/{volume}/_schema_inference/ingest_orders"

target_table = f"{catalog}.{schema}.orders_delta"

orders_schema = StructType([
    StructField("order_id",      StringType(),   True),
    StructField("customer_id",   StringType(),   True),
    StructField("order_ts",      TimestampType(),True),
    StructField("item_id",       StringType(),   True),
    StructField("qty",           IntegerType(),  True),
    StructField("price",         DoubleType(),   True),
])

df = (
    spark.readStream #auto loader
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         #.option("cloudFiles.schemaLocation", schema_loc)  # required for Auto Loader
         .option("header", "true")
         .schema(orders_schema)
         .load(incoming_path)
)

query = (
    df.writeStream
      .format("delta")
      .outputMode("append")
      .option("checkpointLocation", checkpoint_dir)
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .toTable(target_table)
)


In [0]:
select count(1)
from workspace.pyspark_training.orders_delta